# Multi-agent RAG with AutoGen: Build locally with Qwen 2.5

**Original Authors:** Kelly Abuelsaad, Anna Gutowska  
**Adapted by:** Ali Yavari

Can you build [agentic workflows](https://www.ibm.com/think/topics/agentic-workflows) without needing extremely large, costly [large language models (LLMs)](https://www.ibm.com/think/topics/large-language-model)? The answer is yes. In this tutorial — adapted from the original IBM Granite version — we will demonstrate how to build a multi-agent RAG system **fully locally** with AutoGen and an open-source [Qwen 2.5](https://qwenlm.github.io/blog/qwen2.5/) 7B model served by [Ollama](https://ollama.com/). The setup is tuned to run comfortably on an Apple Silicon MacBook (M-series) with 16 GB of unified memory.

## Agentic RAG overview
[Retrieval-augmented generation (RAG)](https://www.ibm.com/think/topics/retrieval-augmented-generation) is an effective way of providing an LLM with additional datasets from various data sources without the need for expensive fine-tuning. Similarly, [agentic RAG](https://www.ibm.com/think/topics/agentic-rag) leverages an [AI agent](https://www.ibm.com/think/topics/ai-agents)’s ability to plan and execute subtasks along with the retrieval of relevant information to supplement an LLM's knowledge base. This ability allows for the optimization and greater scalability of RAG applications compared to traditional [chatbots](https://www.ibm.com/think/topics/chatbots). No longer do we need to write complex SQL queries to extract relevant data from a knowledge base.

The future of agentic RAG is multi-agent RAG, where several specialized agents [collaborate](https://www.ibm.com/think/topics/multi-agent-collaboration) to achieve optimal latency and efficiency. We will demonstrate this collaboration by using a small, efficient model — **Qwen 2.5 7B Instruct** — and combining it with a modular agent architecture. We will use multiple specialized "mini agents" that collaborate to achieve tasks through adaptive planning and tool or function calling. Like humans, a team of agents, or a [multi-agent system](https://www.ibm.com/think/topics/multiagent-system), often outperforms the heroic efforts of an individual, especially when they have clearly defined roles and effective [communication](https://www.ibm.com/think/topics/ai-agent-communication).

For the [orchestration](https://www.ibm.com/think/topics/ai-agent-orchestration) of this collaboration, we use AutoGen (AG2) as the core [framework](https://www.ibm.com/think/insights/top-ai-agent-frameworks) to manage workflows and decision-making, alongside Ollama for local LLM serving and Open WebUI for interaction. AutoGen is a framework for creating multi-agent AI applications developed by Microsoft.<sup>1</sup> Every component leveraged in this tutorial is [open source](https://www.ibm.com/think/topics/open-source). Together, these tools enable you to build an AI system that is both powerful and privacy-conscious, without leaving your laptop.

## Why Qwen 2.5 7B on a MacBook M4 (16 GB)?
The original IBM tutorial uses Granite 3.2 8B. For a MacBook M4 with 16 GB of unified memory, **Qwen 2.5 7B Instruct** is a stronger fit:

| Capability | Granite 3.2 8B | **Qwen 2.5 7B** |
|---|---|---|
| Disk size (Q4_K_M) | ~5.0 GB | **~4.7 GB** |
| Context window | 8K | **32K** |
| Native tool / function calling | Limited | **First-class** |
| Reasoning & math benchmarks | Solid | **Often higher at the same size** |
| Apple Silicon (Metal) inference speed | Good | **Very good** |
| License | Apache 2.0 | **Apache 2.0** |

For multi-agent RAG, **tool calling reliability is the single most important factor** — the planner, research assistant, step critic, goal judge, reflection, and report generator agents all rely on structured outputs. Qwen 2.5 was explicitly trained for agentic workflows and tool use, which translates to fewer malformed function calls and more stable end-to-end runs.

> **Alternatives** that also fit 16 GB:
> - `llama3.1:8b` — strong native tool calling, very well tested.
> - `mistral-nemo:12b-instruct-2407` — wider context, slightly heavier (~7 GB).
> - `qwen2.5:14b` — possible at Q4 (~9 GB), but leaves less headroom.

## Multi-agent architecture: When collaboration beats competition
Our retrieval agent relies on a modular architecture in which each agent has a specialized role. Like humans, agents perform best when they have targeted instructions and just enough context to make an informed decision. Too much extraneous information, such as an unfiltered chat history, can create a “needle in the haystack” problem, where it becomes increasingly difficult to decipher signal from noise.

In this [agentic AI architecture](https://www.ibm.com/think/topics/agentic-architecture), the agents work together sequentially to achieve the goal. Here is how the generative AI system is organized:

**Planner agent**: Creates the initial high-level plan, once in the beginning of the workflow. For example, if a user asks, “What are comparable open source projects to the ones my team is using?” then, the agent will put together a step-by-step plan that might look something like this: “1. Search team documents for open source technologies. 2. Search the web for similar open source projects to the ones found in step 1.” If any of these steps fail or provide insufficient results, the steps can be later adapted by the reflection agent.

**Research Aasistant**: The research assistant is the workhorse of the system. It takes in and executes instructions such as “Search team documents for open source technologies.” For step 1 of the plan, it uses the initial instruction from the planner agent. For subsequent steps, it also receives curated context from the outcomes of previous steps. 

For example, if asked to “Search the web for similar open source projects,” it will also receive the output from the previous document search step. Depending on the instruction, the research assistant can use tools like web search or document search, or both, to fulfill its task. 

**Step critic**: The step critic is responsible for deciding whether the output of the previous step satisfactorily fulfilled the instruction it was given. It receives two pieces of information: the single-step instruction that was just executed and the output of that instruction. Having a step critic weigh in on the conversation brings clarity around whether the goal was achieved, which is needed for the planning of the next step. 

**Goal judge**: The goal judge determines whether the ultimate objective has been met, based on all of the requirements of the provided goal, the plans drafted to achieve it, and the information gathered so far. The output of the judge is either "YES" or "NOT YET" followed by a brief explanation that is no longer than one or two sentences.

**Reflection agent**: The reflection agent is our executive decision-maker. It decides what step to take next, whether that is encroaching onto the next planned step, pivoting course to make up for mishaps or confirming that the goal has been completed. Like a real-life CEO, it performs its best decision-making when it has a clear goal in mind and is presented with concise findings on the progress that has or has not been made to reach that goal. The output of the reflection agent is either the next step to take or the instructions to terminate if the goal has been reached. We present the reflection agent with the following items:
- The goal
- The original plan
- The last step that was executed
- The result of the last step indicating success or failure
- A concise sequence of previously executed instructions (just the instructions, not their output)

Presenting these items in a structured format makes it clear to our decision maker what has been done so that it can decide what needs to happen next. 

**Report Generator**: Once the goal is achieved, the Report Generator synthesizes all findings into a cohesive output that directly answers the original query. While each step in the process generates targeted outputs, the Report Generator ties everything together into a final report.

## Leveraging open source tools
For beginners, it can be difficult to build an agentic AI application from scratch. Hence, we will use a set of open source tools. Our local Retrieval Agent integrates multiple tools for agentic RAG.

**Open WebUI**: The user interacts with the system through an intuitive chat interface hosted in Open WebUI. This interface acts as the primary point for submitting queries (such as “Fetch me the latest news articles pertaining to my project notes”) and viewing the outputs.

**Python-based agent (AG2 framework)**: At the core of the system is a Python-based agent built by using AutoGen (AG2). This agent coordinates the workflow by breaking down tasks and dynamically calling tools to execute steps.

The agent has access to two primary tools:

- Document search tool: Fetches relevant information from a vector database containing uploaded project notes or documents stored as embeddings. This vector search leverages the built-in documental retrieval APIs inside Open WebUI, rather than setting up an entirely separate data store.

- Web search tool: Performs web-based searches to gather external knowledge and real-time information. In this case, we are using SearXNG as our metasearch engine.

**Ollama**: The Qwen 2.5 7B Instruct LLM serves as the language model powering the system. It is hosted locally with Ollama, ensuring fast inference, cost efficiency and data privacy. On a MacBook with Apple Silicon, Ollama leverages the Metal GPU backend out of the box. If you'd like to run this project with larger hosted models, API access through IBM [watsonx.ai®](https://www.ibm.com/products/watsonx-ai) or OpenAI is also possible — this would, however, require an API key. In this tutorial we stay fully local.

Other common open source, agent frameworks not covered in this tutorial include [LangChain](https://www.ibm.com/think/topics/langchain), [LangGraph](https://www.ibm.com/think/topics/langgraph) and [crewAI](https://www.ibm.com/think/topics/crew-ai).

## Steps
Detailed setup instructions as well as the entire project can be viewed on the [IBM Granite Community GitHub](https://github.com/ibm-granite-community/granite-retrieval-agent). The Jupyter Notebook version of this tutorial can be found on [GitHub](https://github.com/IBM/ibmdotcom-tutorials) as well.

The following steps provide a quick setup for the local Retrieval agent.

### Step 1: Install Ollama and pull Qwen 2.5
Installing Ollama is as simple as downloading the client from the [official Ollama site](https://ollama.com/). On macOS you can also install via Homebrew:

```sh
brew install --cask ollama
```

After installing, start the Ollama service (the macOS app starts a background daemon on `localhost:11434`) and pull the Qwen 2.5 7B Instruct model:

```sh
ollama pull qwen2.5:7b
```

Optional — pull a small embedding model so the RAG pipeline can use a fully local embedder later:

```sh
ollama pull nomic-embed-text
```

You are now up and running with Ollama and Qwen 2.5.

### Step 2. Build a simple agent (optional)

Before we begin the setup of the complete multi-agent RAG project, let’s unpack a simpler example. To continue, set up a Jupyter Notebook in your preferred integrated development environment (IDE) and activate a virtual environment by running the following commands in your terminal. 

```sh
python3.11 -m venv venv
source venv/bin/activate
```

We'll need a few libraries for this simple agent: AutoGen (AG2) with the Ollama provider, ChromaDB for the vector store, sentence-transformers for embeddings, and `requests` so we can sanity-check the local Ollama server.

In [ ]:
!pip install -qU langchain chromadb tf-keras pyautogen "ag2[ollama]" sentence_transformers requests

In [ ]:
import requests
from autogen.agentchat.contrib.retrieve_assistant_agent import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

### Module: configuration

We centralize all model and endpoint settings in one place so that every agent in the rest of the notebook reads from the same source of truth. The `make_llm_config()` factory returns a fresh config dict so we can safely tweak per-agent fields (e.g. temperature) without mutating shared state.

In [ ]:
OLLAMA_HOST = "http://localhost:11434"
TASK_MODEL = "qwen2.5:7b"
EMBED_MODEL = "nomic-embed-text"


def make_llm_config(model: str = TASK_MODEL, temperature: float = 0.0) -> dict:
    """Return a fresh AutoGen llm_config dict pointing at our local Ollama server.

    A factory (rather than a shared dict) avoids accidental cross-agent mutation
    when an individual agent wants to override a setting like temperature.
    """
    return {
        "config_list": [
            {
                "model": model,
                "api_type": "ollama",
                "client_host": OLLAMA_HOST,
            }
        ],
        "temperature": temperature,
        "cache_seed": None,
    }


ollama_llm_config = make_llm_config()
ollama_llm_config

### Module: Ollama health check

Before instantiating any agents, let's verify that the Ollama daemon is actually reachable and that the Qwen model is available. Catching this early gives a much clearer error than letting AutoGen fail mid-conversation.

In [ ]:
def check_ollama(host: str = OLLAMA_HOST, required_models: list[str] | None = None) -> None:
    """Confirm that Ollama is running and that required models are pulled.

    Raises a RuntimeError with an actionable message if anything is missing.
    """
    required_models = required_models or [TASK_MODEL]
    try:
        resp = requests.get(f"{host}/api/tags", timeout=3)
        resp.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {host}. Is the Ollama app running? "
            f"Original error: {exc}"
        ) from exc

    available = {m["name"] for m in resp.json().get("models", [])}
    missing = [m for m in required_models if m not in available and f"{m}:latest" not in available]
    if missing:
        raise RuntimeError(
            f"Ollama is running but these models are not pulled yet: {missing}. "
            f"Run e.g. `ollama pull {missing[0]}`."
        )

    print(f"Ollama OK at {host}. Available models: {sorted(available)}")


check_ollama()

### Module: agent factories

These two helpers wrap AutoGen's verbose agent constructors so the rest of the notebook reads like a high-level recipe. Each builder accepts overrides where useful (system message, model, retrieval config), but applies sensible defaults derived from the configuration module above.

In [ ]:
def build_assistant(
    name: str = "assistant",
    system_message: str = "You are a helpful assistant.",
    model: str = TASK_MODEL,
    temperature: float = 0.0,
) -> AssistantAgent:
    """Create a generic AutoGen AssistantAgent backed by our local Ollama model."""
    return AssistantAgent(
        name=name,
        system_message=system_message,
        llm_config=make_llm_config(model=model, temperature=temperature),
    )


def build_rag_proxy(
    docs_path: str | list[str],
    collection_name: str = "autogen_docs",
    name: str = "ragproxyagent",
    max_consecutive_auto_reply: int = 3,
    extra_retrieve_config: dict | None = None,
) -> RetrieveUserProxyAgent:
    """Create a RetrieveUserProxyAgent that ingests `docs_path` into a Chroma collection."""
    retrieve_config = {
        "task": "qa",
        "docs_path": docs_path,
        "get_or_create": True,
        "collection_name": collection_name,
        "overwrite": True,
    }
    if extra_retrieve_config:
        retrieve_config.update(extra_retrieve_config)

    return RetrieveUserProxyAgent(
        name=name,
        max_consecutive_auto_reply=max_consecutive_auto_reply,
        is_termination_msg=lambda msg: msg.get("content") is not None
        and "TERMINATE" in (msg.get("content") or ""),
        system_message="Context retrieval assistant.",
        retrieve_config=retrieve_config,
        code_execution_config=False,
        human_input_mode="NEVER",
    )

### Run a simple RAG demo

With the configuration, health check, and factory modules in place, building an agent is now a one-liner. We start with a generic `AssistantAgent` powered by Qwen 2.5.

In [ ]:
assistant = build_assistant(
    name="assistant",
    system_message="You are a helpful assistant.",
)

This agent uses Qwen 2.5 to synthesize the information returned by the retrieval proxy agent. The document we feed to the RAG agent as additional context is the raw README Markdown from the AutoGen repository on GitHub. The `extra_retrieve_config` parameter on `build_rag_proxy` lets you tweak Chroma- or embedding-related options such as `vector_db`, `chunk_token_size`, and `embedding_model`. For the full list, see the [official documentation](https://microsoft.github.io/autogen/0.2/docs/reference/agentchat/contrib/retrieve_user_proxy_agent/).

In [ ]:
ragproxyagent = build_rag_proxy(
    docs_path="https://raw.githubusercontent.com/microsoft/autogen/main/README.md",
    collection_name="autogen_docs",
)

Now, we can initiate a chat with our RAG agent to ask a question that pertains to the document provided as context. 

In [ ]:
qs = "What languages does AutoGen support?"
result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qs
)  

print(result)

Trying to create collection.


2025-07-21 12:20:36,125 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 1 chunks.
2025-07-21 12:20:36,129 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.


VectorDB returns doc_ids:  [['8e9131c7']]
Adding content of doc 8e9131c7 to context.
ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: What languages does AutoGen support?

Context is: <a name="readme-top"></a>

<div align="center">
<img src="https://microsoft.github.io/autogen/0.2/img/ag.svg" alt="AutoGen Logo" width="100">

[![Twitter](https://img.shields.io/twitter/url/https/twitter.com/cloudposse.svg?style=social&label=Follow%20%40pyautogen)](https://twitter.com/pyautogen)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Company?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/company/105812540)
[![Discord](https://img.shields.io/badge/discord-chat-green?logo=discord)](https

Great! Our assistant agent and RAG agent successfully synthesized the additional context to correctly respond to the user query with the programming languages currently supported by AutoGen. You can think of this as a group chat between agents exchanging information. This is a simple, fully local demonstration of agentic RAG with AutoGen and Qwen 2.5 — running entirely on your MacBook.

### Step 3. Install Open WebUI

Now, let’s move on to building a more advanced agentic RAG system. In your terminal, install and run Open WebUI.

```sh 
pip install open-webui
open-webui serve
```

**Footnotes**: 

<sup>1</sup>  Wu, Qingyun, et al. “AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation Framework.” *GitHub*, 2023, github.com/microsoft/autogen.